# MNIST MLP3 — SGD + Nesterov vs Full Matrix-Log RG

This notebook uses the exact qualified MNIST/MLP3 SGD baseline recipe and adds the Full Matrix-Log RG wrapper only to `fc1.weight` and `fc2.weight`. The official test set is never evaluated during hyperparameter selection. The final comparison uses the same three baseline seeds, restartable checkpoints, run-level 95% Student-t intervals, and direct WeightWatcher `alpha`, `ERG_gap`, and `num_traps` measurements.

The optimizer has two scientifically distinct modes:

- `radial`: removes only net decrease of `Phi_R = ||log X_tilde_R||_F^2/(2m)`.
- `modewise`: removes every first-order retained log-eigenvalue motion toward zero. This is the literal full matrix condition.

For Mac execution, the retained right basis is refreshed once per epoch and cached. Corrections diagonalize only the retained covariance; they do **not** recompute a full layer SVD inside every training step.


In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO = None
for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "baseline" / "rg_baselines").is_dir() and (
        path / "optimizers" / "full_matrix_log_rg" / "full_matrix_log_rg"
    ).is_dir():
        REPO = path.resolve()
        break
if REPO is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers")

for path in [REPO / "baseline", REPO / "optimizers" / "full_matrix_log_rg"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    summarize_numeric_metrics,
)
from rg_baselines.engine import choose_device
from full_matrix_log_rg import FullMatrixLogConfig
from full_matrix_log_rg.experiment import run_mnist_sgd, run_validation_grid

DEVICE = choose_device()
DATA_DIR = Path(
    os.environ.get("RG_BASELINE_DATA_DIR", Path.home() / "rg-optimizer-data")
).expanduser().resolve()
RUN_ROOT = Path(
    os.environ.get(
        "RG_FML_RUN_ROOT", Path.home() / "rg-optimizer-runs" / "full_matrix_log_rg"
    )
).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({"repo": str(REPO), "device": str(DEVICE), "data": str(DATA_DIR), "runs": str(RUN_ROOT)})


## Preflight

This validates the core geometry, the stronger modewise projector, rectangular matrices, correction cadence, and optimizer-state restart before starting MNIST.


In [ ]:
environment = dict(os.environ)
environment["PYTHONPATH"] = os.pathsep.join(
    [str(REPO / "optimizers" / "full_matrix_log_rg"), environment.get("PYTHONPATH", "")]
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(REPO / "optimizers" / "full_matrix_log_rg" / "tests"),
        "-v",
    ],
    check=True,
    env=environment,
)


## Exact baseline recipe

The architecture, split, initialization, optimizer, warm-up, cosine decay, clipping, and WeightWatcher settings are inherited from the qualified baseline package. Only the RG wrapper differs.


In [ ]:
BASE_CONFIG = BaselineConfig(
    optimizer="sgd_momentum",
    epochs=30,
    validation_size=5_000,
    sgd_learning_rate=0.05,
    sgd_min_learning_rate=5e-4,
    sgd_warmup_epochs=2,
    sgd_momentum=0.90,
    sgd_dampening=0.0,
    sgd_nesterov=True,
    sgd_weight_decay=1e-4,
    grad_clip_norm=1.0,
    ww_randomize=True,
    save_epoch_checkpoints=True,
)
SEEDS = tuple(DEFAULT_BASELINE_SEEDS)
TARGET_MATRICES = ("fc1.weight", "fc2.weight")
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
BASE_CONFIG.validate()
display(pd.DataFrame([asdict(BASE_CONFIG)]))


## Bounded validation-only search

The Mac-default search is deliberately small: two correction geometries, two strengths, and two cadences, with a fixed 10% correction cap. It uses one preregistered seed and five epochs. Set `RG_FML_SKIP_GRID=1` to reuse `selected_config.json`, or set `RG_FML_GRID_EPOCHS` to change only the search horizon. The test set is not evaluated in these runs.


In [ ]:
GRID_EPOCHS = int(os.environ.get("RG_FML_GRID_EPOCHS", "5"))
if GRID_EPOCHS < 3:
    raise ValueError("RG_FML_GRID_EPOCHS must be at least 3")
GRID_BASE = replace(
    BASE_CONFIG,
    seed=int(SEEDS[0]),
    epochs=GRID_EPOCHS,
    save_epoch_checkpoints=False,
)
GRID_CANDIDATES = [
    FullMatrixLogConfig(
        mode=mode,
        projection_strength=strength,
        max_correction_ratio=0.10,
        apply_every_steps=cadence,
        parameter_names=TARGET_MATRICES,
    )
    for mode in ("radial", "modewise")
    for strength in (0.5, 1.0)
    for cadence in (25, 100)
]
GRID_ROOT = RUN_ROOT / "validation_grid"
selected_path = GRID_ROOT / "selected_config.json"

if os.environ.get("RG_FML_SKIP_GRID", "0") == "1":
    if not selected_path.is_file():
        raise FileNotFoundError(f"RG_FML_SKIP_GRID=1 but {selected_path} does not exist")
    payload = json.loads(selected_path.read_text(encoding="utf-8"))
    payload["parameter_names"] = tuple(payload["parameter_names"]) if payload.get("parameter_names") else None
    BEST = FullMatrixLogConfig(**payload)
    GRID = pd.read_csv(GRID_ROOT / "grid_results_ranked.csv")
else:
    grid_result = run_validation_grid(
        GRID_BASE,
        GRID_CANDIDATES,
        data_dir=DATA_DIR,
        output_dir=GRID_ROOT,
        device=DEVICE,
        progress=True,
        resume=True,
    )
    BEST = grid_result.selected_config
    GRID = grid_result.results

display(GRID)
print("selected:", BEST)


## Final three-seed comparison

Each seed writes latest, best-validation, per-epoch, and final checkpoints. Interrupted runs resume from `checkpoint_latest.pt`; completed runs are loaded without retraining.


In [ ]:
performance_frames = []
spectral_frames = []
correction_frames = []

for seed in SEEDS:
    config = replace(BASE_CONFIG, seed=int(seed))
    baseline = run_mnist_sgd(
        config,
        rg_config=None,
        data_dir=DATA_DIR,
        output_dir=RUN_ROOT / "final" / "sgd_momentum" / f"seed_{seed}",
        device=DEVICE,
        evaluate_test=True,
        progress=True,
        resume=True,
    )
    extended = run_mnist_sgd(
        config,
        rg_config=BEST,
        data_dir=DATA_DIR,
        output_dir=RUN_ROOT / "final" / "full_matrix_log_rg" / f"seed_{seed}",
        device=DEVICE,
        evaluate_test=True,
        progress=True,
        resume=True,
    )
    performance_frames.extend([baseline.performance, extended.performance])
    spectral_frames.extend([baseline.spectral, extended.spectral])
    if not extended.corrections.empty:
        correction_frames.append(extended.corrections)

PERFORMANCE = pd.concat(performance_frames, ignore_index=True)
SPECTRAL = pd.concat(spectral_frames, ignore_index=True, sort=False)
CORRECTIONS = (
    pd.concat(correction_frames, ignore_index=True, sort=False)
    if correction_frames
    else pd.DataFrame()
)

AGGREGATE = RUN_ROOT / "final" / "aggregate"
PLOT_DIR = AGGREGATE / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PERFORMANCE.to_csv(AGGREGATE / "performance_by_epoch_and_seed.csv", index=False)
SPECTRAL.to_csv(AGGREGATE / "spectral_metrics_by_epoch_layer_and_seed.csv", index=False)
CORRECTIONS.to_csv(AGGREGATE / "rg_corrections_by_step.csv", index=False)


## Run-level 95% confidence intervals


In [ ]:
PERFORMANCE_SUMMARY = summarize_numeric_metrics(
    PERFORMANCE,
    group_columns=("run", "epoch"),
    metrics=(
        "train_loss",
        "validation_loss",
        "test_loss",
        "train_accuracy",
        "validation_accuracy",
        "test_accuracy",
    ),
)
SPECTRAL_OK = SPECTRAL[SPECTRAL["status"].eq("ok")].copy()
SPECTRAL_SUMMARY = summarize_numeric_metrics(
    SPECTRAL_OK,
    group_columns=("run", "layer", "epoch"),
    metrics=("alpha", "ERG_gap", "num_traps", "m_midpoint"),
)
PERFORMANCE_SUMMARY.to_csv(AGGREGATE / "performance_summary_95ci.csv", index=False)
SPECTRAL_SUMMARY.to_csv(AGGREGATE / "spectral_summary_95ci.csv", index=False)


REQUIRED_PERFORMANCE_METRICS = {
    "train_loss", "validation_loss", "test_loss",
    "train_accuracy", "validation_accuracy", "test_accuracy",
}
required_performance = PERFORMANCE_SUMMARY[
    PERFORMANCE_SUMMARY["metric"].isin(REQUIRED_PERFORMANCE_METRICS)
]
if set(required_performance["metric"]) != REQUIRED_PERFORMANCE_METRICS:
    raise RuntimeError("Missing required performance summary metrics")
if not required_performance["n"].eq(len(SEEDS)).all():
    raise RuntimeError("Performance confidence intervals do not contain all three runs")

REQUIRED_SPECTRAL_METRICS = {"alpha", "ERG_gap", "num_traps", "m_midpoint"}
required_spectral = SPECTRAL_SUMMARY[
    SPECTRAL_SUMMARY["metric"].isin(REQUIRED_SPECTRAL_METRICS)
]
if set(required_spectral["metric"]) != REQUIRED_SPECTRAL_METRICS:
    raise RuntimeError("Missing required direct WeightWatcher summary metrics")
if not required_spectral["n"].eq(len(SEEDS)).all():
    raise RuntimeError("Spectral confidence intervals do not contain all three runs")

display(PERFORMANCE_SUMMARY.sort_values(["metric", "run", "epoch"]))
display(SPECTRAL_SUMMARY.sort_values(["metric", "layer", "run", "epoch"]))


In [ ]:
for metric in ("validation_accuracy", "test_accuracy", "validation_loss", "test_loss"):
    summary = PERFORMANCE_SUMMARY[PERFORMANCE_SUMMARY["metric"].eq(metric)]
    figure, axis = plt.subplots(figsize=(9, 5))
    for run, group in summary.groupby("run"):
        group = group.sort_values("epoch")
        axis.plot(group["epoch"], group["mean"], linewidth=2.0, label=run)
        axis.fill_between(group["epoch"], group["ci_low"], group["ci_high"], alpha=0.16)
    axis.set(xlabel="Epoch", ylabel=metric.replace("_", " ").title(), title=metric.replace("_", " ").title())
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"{metric}_95ci.png", dpi=170, bbox_inches="tight")
    plt.show()


In [ ]:
for metric in ("alpha", "ERG_gap", "num_traps"):
    summary = SPECTRAL_SUMMARY[SPECTRAL_SUMMARY["metric"].eq(metric)]
    for layer, layer_summary in summary.groupby("layer"):
        figure, axis = plt.subplots(figsize=(9, 5))
        for run, group in layer_summary.groupby("run"):
            group = group.sort_values("epoch")
            axis.plot(group["epoch"], group["mean"], linewidth=2.0, label=run)
            axis.fill_between(group["epoch"], group["ci_low"], group["ci_high"], alpha=0.16)
        if metric == "alpha":
            axis.axhline(2.0, linestyle="--", linewidth=1.0)
        axis.set(xlabel="Epoch", ylabel=metric, title=f"{layer}: {metric}")
        axis.grid(alpha=0.25)
        axis.legend(frameon=False)
        figure.tight_layout()
        figure.savefig(PLOT_DIR / f"{layer}_{metric}_95ci.png", dpi=170, bbox_inches="tight")
        plt.show()


## RG correction audit and required artifacts


In [ ]:
if not CORRECTIONS.empty:
    CORRECTION_SUMMARY = CORRECTIONS.groupby(
        ["mode", "parameter"], as_index=False
    ).agg(
        corrections=("correction_ratio", "size"),
        mean_correction_ratio=("correction_ratio", "mean"),
        max_correction_ratio=("correction_ratio", "max"),
        mean_base_potential_drift=("base_potential_drift", "mean"),
        mean_corrected_potential_drift=("corrected_potential_drift", "mean"),
        mean_base_inward_mode_norm=("base_inward_mode_norm", "mean"),
        mean_corrected_inward_mode_norm=("corrected_inward_mode_norm", "mean"),
        cap_fraction=("correction_capped", "mean"),
    )
    CORRECTION_SUMMARY.to_csv(AGGREGATE / "rg_correction_summary.csv", index=False)
    display(CORRECTION_SUMMARY)

required = [
    GRID_ROOT / "grid_results_ranked.csv",
    GRID_ROOT / "selected_config.json",
    AGGREGATE / "performance_by_epoch_and_seed.csv",
    AGGREGATE / "spectral_metrics_by_epoch_layer_and_seed.csv",
    AGGREGATE / "performance_summary_95ci.csv",
    AGGREGATE / "spectral_summary_95ci.csv",
]
for seed in SEEDS:
    for family in ("sgd_momentum", "full_matrix_log_rg"):
        run_dir = RUN_ROOT / "final" / family / f"seed_{seed}"
        required.extend(
            [
                run_dir / "checkpoint_latest.pt",
                run_dir / "checkpoint_best.pt",
                run_dir / "final_state.pt",
                run_dir / "run_complete.json",
            ]
        )
missing = [path for path in required if not path.is_file()]
if missing:
    raise RuntimeError("Missing required artifacts:\n" + "\n".join(map(str, missing)))
print("verified artifacts:", len(required))
